# Patent Landscaping — Snorkel vs MAS (Colab GPU side)

**Split of work**
- **Local `.venv` (done beforehand):** data pipeline + dedup + negative pool + **MAS labeling (10 keys)** → produces `DataSet/mas/mas_ranked_scores.csv`.
- **This Colab notebook (GPU):** Snorkel labeling (needs Python 3.10/3.11) + **SciBERT fine-tune & eval for BOTH arms**.

Runtime → change runtime type → **GPU (T4)**. SciBERT fine-tune ≈ 20–40 min/arm.

> Prereq: push the repo to GitHub first (the clone below needs it). The MAS output
> `mas_ranked_scores.csv` is git-ignored, so upload it in the MAS cell.

In [ ]:
# 1) clone + install
REPO = 'https://github.com/PassionChicken-Leesuin/Patent_Landscaping_Final.git'
import os
if not os.path.exists('Patent_Landscaping_Final'):
    !git clone $REPO
%cd Patent_Landscaping_Final
!pip -q install snorkel transformers datasets scikit-learn accelerate

In [ ]:
# 2) (re)build processed datasets from raw (dedup + negative pool)
!python -m scripts.build_dataset

In [ ]:
# 3) SNORKEL arm — label the candidate pool with LabelModel
!python -m scripts.run_snorkel

In [ ]:
# 4) MAS output — upload mas_ranked_scores.csv produced locally (10-key run)
import os
os.makedirs('DataSet/mas', exist_ok=True)
from google.colab import files
up = files.upload()                       # choose mas_ranked_scores.csv
for name in up:
    os.replace(name, 'DataSet/mas/mas_ranked_scores.csv')
print('saved DataSet/mas/mas_ranked_scores.csv')

In [ ]:
# 5) downstream: SNORKEL arm (assemble -> fine-tune SciBERT -> eval)
!python -m scripts.run_downstream --arm snorkel --epochs 4

In [ ]:
# 6) downstream: MAS arm
!python -m scripts.run_downstream --arm mas --epochs 4

In [ ]:
# 7) compare
import json
for arm in ['snorkel', 'mas']:
    r = json.load(open(f'outputs/metrics_{arm}.json'))
    print(f"{arm:8s}  macroF1={r['macro_f1']:.4f}  AUC={r['auc']:.4f}  "
          f"P={r['precision']:.4f}  R={r['recall']:.4f}  acc={r['accuracy']:.4f}")
    print('          by level:', {k: v for k, v in r['by_expansion_level'].items()})